# IE Pyrace – DQN Performance Analysis
### Part 2: Comparing Original vs Improved Agent

This notebook analyses and compares two versions of the DQN agent:

| | Original (Part 1) | Improved (Part 2) |
|---|---|---|
| Reward | Sparse (0 every step) | Dense (progress-based) |
| State inputs | Bucketed (÷20, integers 0–10) | Continuous (raw px) |
| Actions | 3 (accelerate, left, right) | 3 or 4 (+ brake) |

The goal is to show that the Part 2 improvements lead to measurably better learning.

---
## 1  Configuration

In [ ]:
# ── Original model (Part 1 – sparse reward, bucketed states) ─────────────────
VERSION_ORIG   = 'DQN_v01'
EPISODE_ORIG   = 3000      # change to your actual checkpoint number
MODEL_ORIG     = f'models_{VERSION_ORIG}/dqn_{EPISODE_ORIG}.pth'
BUFFER_ORIG    = f'models_{VERSION_ORIG}/memory_{EPISODE_ORIG}.npy'

# ── Improved model (Part 2 – dense reward, continuous states) ────────────────
VERSION_NEW    = 'DQN_v02'
EPISODE_NEW    = 3000      # change to your actual checkpoint number
MODEL_NEW      = f'models_{VERSION_NEW}/dqn_{EPISODE_NEW}.pth'
BUFFER_NEW     = f'models_{VERSION_NEW}/memory_{EPISODE_NEW}.npy'

STATE_SIZE     = 5
ACTION_SIZE    = 3   # update to 4 if you added BRAKE

---
## 2  Load episode statistics from replay buffers

Both replay buffers store `(state, action, reward, next_state, done)` tuples.
We reconstruct per-episode step counts and cumulative rewards by slicing on the `done` flag.

In [ ]:
import numpy as np
import pandas as pd

def load_episodes(buffer_path):
    """Load a replay buffer .npy and return a per-episode DataFrame."""
    data     = np.load(buffer_path, allow_pickle=True)
    done_col = data[:, 4].astype(bool)
    rows, ep_steps, ep_rwd, ep_num = [], 0, 0.0, 0
    for i, done in enumerate(done_col):
        ep_steps += 1
        ep_rwd   += float(data[i, 2])
        if done:
            ep_num += 1
            rows.append([ep_num, ep_steps, ep_rwd])
            ep_steps, ep_rwd = 0, 0.0
    df = pd.DataFrame(rows, columns=['episode', 'steps', 'reward'])
    print(f'{buffer_path}: {len(df)} episodes, max reward = {df.reward.max():.0f}')
    return df

df_orig = load_episodes(BUFFER_ORIG)
df_new  = load_episodes(BUFFER_NEW)

---
## 3  Learning curves: Original DQN (sparse reward)

The original agent uses the reward function inherited from the Q-Table implementation:
- `0` reward on every surviving step
- `-10000 + distance` on crash  
- `+10000` on completing the lap

This is an extremely **sparse** reward signal — the agent receives no feedback for the vast majority of timesteps.

In [ ]:
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

def smooth(arr, frac_fine=120, frac_coarse=30):
    n = len(arr)
    w0 = max(5, int(n / frac_fine)  | 1)
    w1 = max(5, int(n / frac_coarse)| 1)
    return savgol_filter(arr, w0, 3), savgol_filter(arr, w1, 1)

r_orig = df_orig.reward.values
s0, s1 = smooth(r_orig)

plt.figure(figsize=(16, 5))
plt.plot(df_orig.episode, s0, label='fine smooth',   color='steelblue')
plt.plot(df_orig.episode, s1, label='coarse smooth', color='navy', alpha=0.6)
plt.xlabel('Episode'); plt.ylabel('Cumulative reward')
plt.title('Original DQN – Learning Curve (sparse reward)')
plt.legend(); plt.tight_layout(); plt.show()

df_orig['z'] = np.where(df_orig.reward > 0,
                        df_orig.reward / df_orig.steps,
                        (10000 + df_orig.reward) / df_orig.steps)
win = max(5, int(len(df_orig) / 15) | 1)
plt.figure(figsize=(16, 5))
plt.plot(df_orig.z.rolling(win).mean(), color='steelblue')
plt.xlabel('Episode'); plt.ylabel('Reward per step')
plt.title('Original DQN – Reward per Action Step')
plt.tight_layout(); plt.show()

### Analysis – Original learning curve

**Overall trend:**  
The reward curve shows [DESCRIBE — e.g. *a largely flat trend for the first X episodes, with only slight improvement by episode Y*]. This is consistent with what we would expect from a sparse reward function: the agent receives a gradient signal almost exclusively when it crashes, which makes it very difficult for the neural network to learn which actions lead to good outcomes in the intermediate steps.

> 📝 *If the curve is completely flat: the agent is likely crashing very quickly every episode and never exploring enough to find positive reward. If it slowly rises: the agent is learning to survive longer but not necessarily driving well.*

**Reward per step:**  
The normalised reward-per-step metric shows [DESCRIBE — e.g. *very little improvement, remaining near X throughout training*]. This confirms the agent is not improving its per-action efficiency — it is not learning to make better decisions at each timestep, only occasionally getting lucky with distance before crashing.

**Key problem:**  
The core issue is that with a reward of `0` on every surviving step, the DQN's replay buffer is filled predominantly with transitions that carry no learning signal. The network cannot distinguish between a step that brought the car closer to the checkpoint and one that moved it further away — both return reward `0`. This motivates the dense reward redesign in Part 2.

---
## 4  Learning curves: Improved DQN (dense reward)

The improved agent uses a redesigned reward function that gives feedback **every step**:
- `+1` when moving closer to the next checkpoint
- `-1` when moving away
- `+500` on reaching a checkpoint
- `-10000 + distance` on crash
- `+10000` on completing the lap

This means every transition in the replay buffer carries a meaningful signal.

In [ ]:
r_new = df_new.reward.values
s0n, s1n = smooth(r_new)

plt.figure(figsize=(16, 5))
plt.plot(df_new.episode, s0n, label='fine smooth',   color='darkorange')
plt.plot(df_new.episode, s1n, label='coarse smooth', color='saddlebrown', alpha=0.6)
plt.xlabel('Episode'); plt.ylabel('Cumulative reward')
plt.title('Improved DQN – Learning Curve (dense reward)')
plt.legend(); plt.tight_layout(); plt.show()

df_new['z'] = np.where(df_new.reward > 0,
                       df_new.reward / df_new.steps,
                       (10000 + df_new.reward) / df_new.steps)
win = max(5, int(len(df_new) / 15) | 1)
plt.figure(figsize=(16, 5))
plt.plot(df_new.z.rolling(win).mean(), color='darkorange')
plt.xlabel('Episode'); plt.ylabel('Reward per step')
plt.title('Improved DQN – Reward per Action Step')
plt.tight_layout(); plt.show()

### Analysis – Improved learning curve

**Overall trend:**  
The improved agent shows [DESCRIBE — e.g. *a clear upward trend beginning around episode X, with rewards stabilising near Y by the end of training*]. The earlier onset of learning compared to the original agent reflects the denser reward signal: the network receives a gradient update signal on every single step, allowing it to learn the relationship between radar readings and good actions much faster.

> 📝 *If the curve rises then becomes noisy or drops: this can indicate the agent has found a local optimum (e.g. circling near one checkpoint) rather than learning to complete the full lap. This is a known risk with dense rewards — the agent may exploit the step reward without making real progress.*

**Reward per step:**  
The per-step metric shows [DESCRIBE — e.g. *a steady increase from X to Y over the course of training, confirming the agent is making better decisions at each individual timestep, not just surviving longer by chance*].

**Note on scale:**  
The absolute reward values are not directly comparable to the original agent because the reward function has changed. The important comparison is the **shape of the learning curve** (trend and convergence speed), not the raw numbers.

---
## 5  Direct comparison

To fairly compare convergence speed we normalise both reward curves to the range [0, 1] before plotting them together.

In [ ]:
def normalise(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-8)

_, s1_orig = smooth(r_orig)
_, s1_new  = smooth(r_new)

n_orig = normalise(s1_orig)
n_new  = normalise(s1_new)

plt.figure(figsize=(16, 6))
plt.plot(df_orig.episode, n_orig, label='Original (sparse reward)', color='steelblue')
plt.plot(df_new.episode,  n_new,  label='Improved (dense reward)',  color='darkorange')
plt.xlabel('Episode')
plt.ylabel('Normalised reward (0=worst, 1=best)')
plt.title('Original vs Improved DQN – Normalised Learning Curve Comparison')
plt.legend(); plt.tight_layout(); plt.show()

### Analysis – Comparison

[DESCRIBE THE DIFFERENCE — e.g. *The improved agent (orange) reaches 50% of its maximum reward approximately X episodes earlier than the original agent (blue). The original curve remains flat for the first Y episodes while the improved curve begins rising almost immediately after the replay buffer fills.*]

This difference is directly attributable to the reward function redesign. With the sparse reward the DQN's replay buffer contains mostly zero-reward transitions which provide no useful gradient signal. With the dense reward every transition tells the network whether it moved closer to or further from the next checkpoint, giving a much richer training signal.

The continuous state representation (removing the ÷20 rounding in `observe()`) also contributes: the network can now distinguish between a radar reading of 3.2 px and 3.8 px, which was previously impossible with bucketed states. This gives the agent finer spatial awareness, particularly important for navigating tight corners.

---
## 6  Load both trained models

In [ ]:
try:
    import torch
    import torch.nn as nn
    BACKEND = 'torch'
except ImportError:
    from tensorflow import keras
    BACKEND = 'keras'

def build_model():
    return nn.Sequential(
        nn.Linear(STATE_SIZE, 64), nn.ReLU(),
        nn.Linear(64, 64),         nn.ReLU(),
        nn.Linear(64, ACTION_SIZE),
    )

def get_qvalues(model, states_np):
    if BACKEND == 'torch':
        with torch.no_grad():
            return model(torch.FloatTensor(states_np)).numpy()
    else:
        return model.predict(states_np, verbose=0)

if BACKEND == 'torch':
    model_orig = build_model()
    model_orig.load_state_dict(torch.load(MODEL_ORIG, map_location='cpu'))
    model_orig.eval()

    model_new = build_model()
    model_new.load_state_dict(torch.load(MODEL_NEW, map_location='cpu'))
    model_new.eval()
    print('Both PyTorch models loaded.')
else:
    model_orig = keras.models.load_model(MODEL_ORIG.replace('.pth', '.keras'))
    model_new  = keras.models.load_model(MODEL_NEW.replace('.pth',  '.keras'))
    print('Both Keras/TF models loaded.')

---
## 7  Policy heatmaps

We visualise the learned policy by querying each model over a 2D grid of radar values.
Change `DIM_X` / `DIM_Y` to explore different pairs of radar beams.

In [ ]:
import matplotlib.patches as mpatches

DIM_X = 0
DIM_Y = 2
GRID  = 50

ACTION_LABELS = {0: 'Accelerate', 1: 'Turn Left', 2: 'Turn Right'}
COLOURS       = ['#2ca02c', '#1f77b4', '#d62728']
cmap          = plt.matplotlib.colors.ListedColormap(COLOURS)

def make_grid_states(dim_x, dim_y, grid, state_size):
    x_vals = np.linspace(0, 10, grid)
    y_vals = np.linspace(0, 10, grid)
    xx, yy = np.meshgrid(x_vals, y_vals)
    states = np.full((grid * grid, state_size), 5.0, dtype=np.float32)
    states[:, dim_x] = xx.ravel()
    states[:, dim_y] = yy.ravel()
    return states

grid_states = make_grid_states(DIM_X, DIM_Y, GRID, STATE_SIZE)

for label, model in [('Original', model_orig), ('Improved', model_new)]:
    qvals  = get_qvalues(model, grid_states)
    greedy = qvals.argmax(axis=1).reshape(GRID, GRID)
    max_q  = qvals.max(axis=1).reshape(GRID, GRID)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(greedy, origin='lower', aspect='auto',
                   extent=[0,10,0,10], cmap=cmap, vmin=0, vmax=2)
    axes[0].set_xlabel(f'Radar dim {DIM_X} (px)')
    axes[0].set_ylabel(f'Radar dim {DIM_Y} (px)')
    axes[0].set_title(f'{label} – Greedy action')
    patches = [mpatches.Patch(color=COLOURS[k], label=v)
               for k, v in ACTION_LABELS.items()]
    axes[0].legend(handles=patches, loc='upper right', fontsize=8)

    im1 = axes[1].imshow(max_q, origin='lower', aspect='auto',
                         extent=[0,10,0,10], cmap='viridis')
    axes[1].set_xlabel(f'Radar dim {DIM_X} (px)')
    axes[1].set_ylabel(f'Radar dim {DIM_Y} (px)')
    axes[1].set_title(f'{label} – Max Q-value (confidence)')
    plt.colorbar(im1, ax=axes[1])

    plt.suptitle(f'{label} DQN – dims ({DIM_X},{DIM_Y}), others fixed at 5 px',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

### Analysis – Policy heatmaps

**Original policy:**  
The action map for the original agent shows [DESCRIBE — e.g. *a relatively uniform or noisy pattern, with no clear structure. The agent does not appear to have learned a consistent decision boundary between actions based on these two radar beams.*] The confidence map shows [DESCRIBE — e.g. *low Q-values throughout, indicating the network is uncertain about most states it encounters.*]

> 📝 *A noisy or uniform action map for the original agent is expected given the sparse reward — the network simply hasn't received enough signal to learn meaningful decision boundaries.*

**Improved policy:**  
The improved agent's action map shows [DESCRIBE — e.g. *a much clearer structure: the agent accelerates (green) when both radar beams report high values (open space ahead), and switches to turning when either beam drops below ~3 px (obstacle nearby). This is an intuitively correct policy.*] The confidence map shows [DESCRIBE — e.g. *higher Q-values especially in the top-right region, where the agent has learned that open space warrants confident acceleration.*]

**Key difference:**  
[DESCRIBE — e.g. *The improved agent has learned clearer, more structured decision boundaries compared to the original. This directly reflects the denser reward signal providing more informative gradients during training.*]

---
## 8  Radar-beam Q-value profiles

For each of the five radar beams we sweep its value from 0 to 10 while fixing the others at 5 px, and plot the Q-value of each action. This shows how sensitive each model is to each individual sensor reading.

In [ ]:
SWEEP        = np.linspace(0, 10, 100)
action_names = ['Accelerate', 'Turn Left', 'Turn Right']
colours_act  = ['#2ca02c', '#1f77b4', '#d62728']

for label, model in [('Original', model_orig), ('Improved', model_new)]:
    fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)
    for d in range(5):
        states = np.full((len(SWEEP), STATE_SIZE), 5.0, dtype=np.float32)
        states[:, d] = SWEEP
        qv = get_qvalues(model, states)
        for a in range(ACTION_SIZE):
            axes[d].plot(SWEEP, qv[:, a], color=colours_act[a],
                         label=action_names[a])
        axes[d].set_title(f'Beam {d}')
        axes[d].set_xlabel('Value (px)')
        if d == 0:
            axes[d].set_ylabel('Q-value')
        axes[d].legend(fontsize=7)
    plt.suptitle(f'{label} DQN – Q-value profiles per radar beam\n'
                 '(other beams fixed at 5 px)', fontsize=11)
    plt.tight_layout()
    plt.show()

### Analysis – Radar beam profiles

**Original agent:**  
The Q-value profiles for the original agent are [DESCRIBE — e.g. *largely flat across all five beams, suggesting the network has not learned to meaningfully differentiate actions based on individual radar readings. This is consistent with the sparse reward providing insufficient signal for the network to associate specific sensor patterns with specific outcomes.*]

**Improved agent:**  
The improved agent shows [DESCRIBE — e.g. *more pronounced variation across the beam sweep, particularly in beams X and Y. As beam 2 (front-centre) increases, the Q-value for accelerating rises relative to turning, which is intuitive — a clear path ahead should encourage the car to speed up rather than steer.*]

**Beam-by-beam (improved agent):**
- **Beam 0:** [DESCRIBE]
- **Beam 1:** [DESCRIBE]
- **Beam 2 (front):** [DESCRIBE]
- **Beam 3:** [DESCRIBE]
- **Beam 4:** [DESCRIBE]

> 📝 *If the improved agent still shows flat profiles: the continuous state change may not have had much effect yet, or more training episodes are needed. The reward function change is the primary driver of improvement.*

---
## 9  Summary statistics

In [ ]:
def summary(df, label):
    tail = df.reward.tail(max(1, len(df)//10))
    print(f'--- {label} ---')
    print(f'  Total episodes       : {len(df)}')
    print(f'  Mean reward (last 10%): {tail.mean():.1f}')
    print(f'  Max reward ever      : {df.reward.max():.1f}')
    print(f'  Mean steps/episode   : {df.steps.mean():.1f}')
    print()

summary(df_orig, 'Original DQN (sparse reward)')
summary(df_new,  'Improved DQN (dense reward)')

### Summary table

| Metric | Original DQN | Improved DQN |
|---|---|---|
| Total episodes | [FILL IN] | [FILL IN] |
| Mean reward (last 10%) | [FILL IN] | [FILL IN] |
| Max reward ever | [FILL IN] | [FILL IN] |
| Mean steps per episode | [FILL IN] | [FILL IN] |

---
## 10  Discussion and conclusions

### Why the improvements worked

**1. Dense reward function**  
The single most impactful change was replacing the sparse reward (0 on every surviving step) with a step-level signal based on progress toward the next checkpoint. With the original reward, the DQN's replay buffer was dominated by zero-reward transitions, making it nearly impossible for the network to associate specific states with specific outcomes. The dense reward gives the network a gradient signal at every timestep, dramatically accelerating convergence.

**2. Continuous state inputs**  
Removing the `/20` rounding in `observe()` allows the network to exploit the full resolution of the radar sensors. The original bucketed representation discretised each reading into only 11 possible values (0–10), which is identical to the Q-Table's bucket resolution and wastes the DQN's ability to generalise over continuous space. With raw pixel values the network can learn finer-grained decision boundaries.

[ADD ANY OTHER IMPROVEMENTS YOU IMPLEMENTED — e.g. BRAKE action]

### Remaining limitations and next steps

- **No target network:** the vanilla DQN uses the same network for both prediction and target Q-values, which can cause training instability. Adding a periodically-updated target network (standard in DQN) would likely reduce variance in the learning curve.
- **Fixed epsilon decay:** a more adaptive exploration strategy (e.g. decaying epsilon based on performance rather than episode count) could improve sample efficiency.
- **Reward shaping:** the current dense reward uses a simple closer/further heuristic. More sophisticated shaping (e.g. penalising low speed, rewarding smooth steering) could further improve lap times.
- **Bonus — Stable Baselines 3:** migrating to algorithms like PPO or DDPG via SB3 would allow continuous action spaces (smooth steering angles rather than discrete turn increments) and access to more advanced training infrastructure.